# 🦀 Day 4: Borrowing Rules Deep Dive

Today:
- Multiple `&T` OR one `&mut T` — explored thoroughly
- No dangling references
- Non-Lexical Lifetimes (NLL)
- Common borrowing patterns

---

## 📏 Rule: No Dangling References

In [ ]:
// Rust prevents dangling references at compile time

// This would create a dangling reference:
// fn dangle() -> &String {
//     let s = String::from("hello");
//     &s  // ❌ s is dropped, reference would be dangling!
// }

// Fix: return the owned value
fn no_dangle() -> String {
    let s = String::from("hello");
    s  // Ownership moves to caller — no dangling!
}

let result = no_dangle();
println!("{}", result);

## 🔄 Non-Lexical Lifetimes (NLL)

Rust is smart: a reference's lifetime ends at its **last use**, not at the end of scope.

In [ ]:
let mut data = String::from("hello");

// NLL in action:
let r1 = &data;           // immutable borrow starts
let r2 = &data;           // another immutable borrow
println!("{}, {}", r1, r2); // last use of r1 and r2
// r1 and r2 are now "dead" — their lifetimes end here

let r3 = &mut data;       // ✅ mutable borrow OK — no active immutable borrows
r3.push_str(" world");
println!("{}", r3);

In [ ]:
// Without NLL (old Rust), this wouldn't compile:
let mut v = vec![1, 2, 3];
let first = &v[0];       // immutable borrow
println!("First: {}", first); // last use of first

v.push(4);               // ✅ Now OK because first is dead
println!("v: {:?}", v);

## 🐛 Common Borrowing Mistakes & Fixes

In [ ]:
// Mistake 1: Iterating and modifying simultaneously
let mut scores = vec![85, 92, 78, 95, 88];

// BAD: Can't mutate while iterating
// for score in &scores {
//     if *score < 80 {
//         scores.push(0);  // ❌ Mutating while borrowed!
//     }
// }

// FIX: Collect indices first, then modify
let low_indices: Vec<usize> = scores.iter()
    .enumerate()
    .filter(|(_, &s)| s < 80)
    .map(|(i, _)| i)
    .collect();

for &i in &low_indices {
    println!("Low score at index {}: {}", i, scores[i]);
}
println!("Scores: {:?}", scores);

In [ ]:
// Mistake 2: Returning reference to local variable
// fn bad_ref() -> &Vec<i32> {
//     let v = vec![1, 2, 3];
//     &v  // ❌ v is dropped, reference dangles
// }

// FIX: Return the owned value
fn good_vec() -> Vec<i32> {
    vec![1, 2, 3]  // Ownership transfers to caller
}

let v = good_vec();
println!("{:?}", v);

In [ ]:
// Mistake 3: Borrowing from HashMap while modifying it
use std::collections::HashMap;

let mut map = HashMap::new();
map.insert("a", 1);
map.insert("b", 2);

// BAD:
// let val = map.get("a").unwrap(); // immutable borrow
// map.insert("c", 3);              // ❌ mutable borrow while immutable exists
// println!("{}", val);

// FIX: Use the immutable borrow before mutating
let val = *map.get("a").unwrap(); // Copy the value out
map.insert("c", 3);               // ✅ No active borrows
println!("a={}, map={:?}", val, map);

## ✅ Borrowing Patterns That Work

In [ ]:
// Pattern 1: Read → Process → Modify
let mut data = vec![1, 2, 3, 4, 5];

// Phase 1: Read (immutable borrow)
let sum: i32 = data.iter().sum();
let avg = sum as f64 / data.len() as f64;
println!("Average: {:.1}", avg);

// Phase 2: Modify (mutable borrow — safe, immutable borrows done)
for x in data.iter_mut() {
    *x = (*x as f64 - avg).round() as i32;
}
println!("Normalized: {:?}", data);

In [ ]:
// Pattern 2: Split borrows on different fields
struct Player {
    name: String,
    health: i32,
    score: u32,
}

let mut player = Player {
    name: String::from("Hero"),
    health: 100,
    score: 0,
};

// Borrowing DIFFERENT fields simultaneously is OK!
let name_ref = &player.name;      // immutable borrow of name
let health_ref = &mut player.health; // mutable borrow of health — different field!
*health_ref -= 10;
println!("{}: health={}", name_ref, health_ref);

## 🏋️ Exercises

In [ ]:
// Exercise 1: Fix this code (multiple solutions exist)
/*
let mut words = vec!["hello".to_string(), "world".to_string()];
let first = &words[0];
words.push("rust".to_string());
println!("First: {}", first);
*/

// YOUR CODE HERE


In [ ]:
// Exercise 2: Write a function that takes &mut Vec<i32>
// and removes all negative numbers, returning count removed

// YOUR CODE HERE


## 🎯 Key Takeaways

1. Rust prevents dangling references at **compile time**
2. **NLL**: reference lifetimes end at last use, not at scope end
3. Collect first, then modify — don't iterate and mutate simultaneously
4. Different struct fields can be borrowed independently
5. Copy values out of borrows to release the borrow early

---
**Tomorrow:** Slices! 🍕